# Prithvi-100M — Wildfire Burn Scar Segmentation

**Environment**: Colab A100 (GPU required)

End-to-end training pipeline. Architecture: Prithvi-EO-1.0-100M backbone + multi-scale FPN neck + FPN decoder.

---

## Sections

| Cells | Section | Description |
|---|---|---|
| 1–2 | Setup | Install, Drive mount |
| 3–5 | Imports, Paths, Dataset | Constants, hyperparameters, data loaders |
| 6 | Architecture | MultiScaleNeck + FPNDecoder + PrithviSegModelV2 |
| 7–8 | **Phase 1** | 25 epochs, backbone frozen, neck + decoder learn |
| 9–10 | **Phase 2** | 20 epochs, last 2 backbone blocks unfrozen |
| 11–12 | Evaluation | Threshold optimization + portfolio figure |

> **Quick eval only** (skip training): run 1–6, load checkpoint, run 11–12

In [ ]:
# [1] Install dependencies
import subprocess, sys, os

needs = False
try:
    import numpy as np
    assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0)
    from terratorch.registry import BACKBONE_REGISTRY
    print(f'OK — numpy={np.__version__}, terratorch ready.')
except Exception as e:
    needs = True
    print(f'Installing ({e})...')

if needs:
    subprocess.run([sys.executable,'-m','pip','install','-q','numpy==2.0.2','--force-reinstall'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'terratorch','einops','timm','segmentation-models-pytorch'], check=True)
    print('Restarting kernel...')
    os.kill(os.getpid(), 9)

In [ ]:
# [2] Drive mount & environment
try:
    import google.colab; IN_COLAB = True
    from google.colab import drive; drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
print(f'IN_COLAB = {IN_COLAB}')

In [ ]:
# [3] Imports
import os, random, subprocess, warnings, time
from pathlib import Path
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from terratorch.registry import BACKBONE_REGISTRY

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch={torch.__version__}  device={DEVICE}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# [4] Paths & constants
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
else:
    BASE = Path('G:/Mon Drive/GeoAI/wildfire-spread')

CKPT    = BASE / 'models/best_prithvi_burn_scar_wildfire.pth'
RESULTS = BASE / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    LOCAL = Path('/content/patches')
    LOCAL.mkdir(exist_ok=True)
    if not (LOCAL / 'images').exists():
        print('Copying patches to /content/ ...')
        subprocess.run(['cp','-r', str(BASE/'data/patches/images'),     str(LOCAL)], check=True)
        subprocess.run(['cp','-r', str(BASE/'data/patches/masks_dnbr'), str(LOCAL)], check=True)
        print('Done.')
    IMG_DIR  = LOCAL / 'images'
    MASK_DIR = LOCAL / 'masks_dnbr'
else:
    IMG_DIR  = BASE / 'data/patches/images'
    MASK_DIR = BASE / 'data/patches/masks_dnbr'

# Prithvi normalization stats (B02,B03,B04,B8A,B11,B12)
BAND_IDX     = [0, 1, 2, 4, 5, 6]
PRITHVI_MEAN = np.array([0.033349,0.057012,0.058897,0.232325,0.197285,0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691,0.026808,0.040041,0.077917,0.087087,0.072420], dtype=np.float32)
IMG_SIZE  = 224
T         = (256 - IMG_SIZE) // 2
VAL_FRAC  = 0.2
BATCH     = 8
FIRE_W    = 5.0

img_paths  = sorted(IMG_DIR.glob('*.npy'))
mask_paths = sorted(MASK_DIR.glob('*.npy'))
print(f'Patches: {len(img_paths)}')

In [ ]:
# [5] Dataset class
class BurnScarDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False):
        self.imgs, self.masks, self.augment = img_paths, mask_paths, augment

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        raw  = np.load(self.imgs[idx],  mmap_mode='r').astype(np.float32)
        mask = np.load(self.masks[idx], mmap_mode='r')
        img  = (raw[BAND_IDX]/10000.0 - PRITHVI_MEAN[:,None,None]) / PRITHVI_STD[:,None,None]
        img  = img[:,  T:T+IMG_SIZE, T:T+IMG_SIZE]
        mask = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy().astype(np.float32)
        img_t, mask_t = torch.from_numpy(img), torch.from_numpy(mask).unsqueeze(0)
        if self.augment:
            if torch.rand(1)>0.5: img_t,mask_t = TF.hflip(img_t),TF.hflip(mask_t)
            if torch.rand(1)>0.5: img_t,mask_t = TF.vflip(img_t),TF.vflip(mask_t)
            k = torch.randint(0,4,(1,)).item()
            if k: img_t,mask_t = TF.rotate(img_t,90*k),TF.rotate(mask_t,90*k)
        return img_t, mask_t

# Train/val split
print('Scanning fire flags...')
fire_flags = np.array([np.load(p,mmap_mode='r').max()>0 for p in tqdm(mask_paths)], dtype=np.int32)
train_idx, val_idx = train_test_split(np.arange(len(img_paths)),
                                       test_size=VAL_FRAC, stratify=fire_flags, random_state=SEED)
val_imgs   = [img_paths[i]  for i in val_idx]
val_masks  = [mask_paths[i] for i in val_idx]
w = torch.tensor([FIRE_W if f else 1.0 for f in fire_flags[train_idx]], dtype=torch.float)
sampler      = WeightedRandomSampler(w, len(w), replacement=True)
train_loader = DataLoader(
    BurnScarDataset([img_paths[i] for i in train_idx], [mask_paths[i] for i in train_idx], augment=True),
    batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(
    BurnScarDataset(val_imgs, val_masks, augment=False),
    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Val batches: {len(val_loader)}')

---

## Architecture — v1.5

Multi-scale neck fuses transformer layers 2, 5, 8, 11 via top-down lateral connections, giving the decoder richer multi-resolution context compared to using only the last layer.

In [ ]:
# [6] Architecture — MultiScaleNeck + FPNDecoder + PrithviSegModelV2
class MultiScaleNeck(nn.Module):
    def __init__(self, n_side=14, d_model=768, fpn_ch=256):
        super().__init__()
        self.n_side = n_side
        self.lateral = nn.ModuleList([
            nn.Sequential(nn.Conv2d(d_model, fpn_ch, 1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(4)
        ])
        self.td_conv = nn.ModuleList([
            nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, padding=1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(3)
        ])
    def tok2map(self, tokens):
        B, L, D = tokens.shape
        return tokens.permute(0, 2, 1).reshape(B, D, self.n_side, self.n_side)
    def forward(self, layers_out):
        feats = [self.lateral[i](self.tok2map(layers_out[idx][:, 1:, :]))
                 for i, idx in enumerate([2, 5, 8, 11])]
        out = feats[3]
        out = self.td_conv[0](feats[2] + out)
        out = self.td_conv[1](feats[1] + out)
        out = self.td_conv[2](feats[0] + out)
        return out  # (B, 256, 14, 14)


class FPNDecoder(nn.Module):
    def __init__(self, in_ch=256):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviSegModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100')
        self.neck     = MultiScaleNeck(n_side=14, d_model=768, fpn_ch=256)
        self.decoder  = FPNDecoder(in_ch=256)
    def forward(self, x):
        return self.decoder(self.neck(self.backbone(x.unsqueeze(2))))

dice_loss  = smp.losses.DiceLoss(mode='binary')
focal_loss = smp.losses.FocalLoss(mode='binary')

def criterion(logits, targets):
    return dice_loss(logits, targets) + focal_loss(logits, targets)

def fire_iou(logits, target, threshold=0.525):
    pred = (logits.sigmoid() > threshold).float()
    if target.sum() == 0 and pred.sum() == 0: return None
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()
    return (tp / (tp + fp + fn + 1e-6)).item()

print('Architecture defined.')

---

## Phase 1 — Backbone Frozen (25 epochs)

Backbone weights transferred from v1.3 checkpoint. Neck + decoder learn from scratch with backbone kept frozen. LR=5e-4, CosineAnnealingLR.

In [ ]:
# [7] Phase 1 setup — transfer v1.3 backbone weights
model = PrithviSegModelV2().to(DEVICE)

# Transfer backbone weights from v1.3 checkpoint
v13_state = torch.load(CKPT, map_location=DEVICE)
backbone_weights = {k: v for k, v in v13_state.items() if k.startswith('backbone.')}
missing, unexpected = model.load_state_dict(backbone_weights, strict=False)
print(f'Backbone weights loaded: {len(backbone_weights)} tensors')
print(f'Neck + decoder initialized from scratch: {len(missing)} tensors')

# Freeze backbone
for p in model.parameters(): p.requires_grad = False
for p in model.neck.parameters(): p.requires_grad = True
for p in model.decoder.parameters(): p.requires_grad = True

LR_P1     = 5e-4
NUM_EP_P1 = 25
optimizer_p1 = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR_P1, weight_decay=1e-4)
scheduler_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1, T_max=NUM_EP_P1, eta_min=1e-5)
print(f'Phase 1: {NUM_EP_P1} epochs | LR={LR_P1} | backbone frozen')

In [ ]:
# [8] Phase 1 training loop
best_iou = 0.0
history_p1 = {'train_loss': [], 'val_loss': [], 'val_iou': []}

for epoch in range(1, NUM_EP_P1 + 1):
    model.train(); model.backbone.eval()
    ep_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'P1 Epoch {epoch:02d}/{NUM_EP_P1} [train]', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer_p1.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_p1.step()
        ep_loss += loss.item()
    scheduler_p1.step()
    history_p1['train_loss'].append(ep_loss / len(train_loader))

    model.eval()
    v_loss, v_iou_list = 0.0, []
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'P1 Epoch {epoch:02d}/{NUM_EP_P1} [val]', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            v_loss += criterion(preds, masks).item()
            iou = fire_iou(preds, masks)
            if iou is not None: v_iou_list.append(iou)
    history_p1['val_loss'].append(v_loss / len(val_loader))
    history_p1['val_iou'].append(np.mean(v_iou_list) if v_iou_list else 0.0)

    print(f'Phase 1 Epoch {epoch:02d} | Train: {history_p1["train_loss"][-1]:.4f} | '
          f'Val: {history_p1["val_loss"][-1]:.4f} | IoU: {history_p1["val_iou"][-1]:.4f}')

    if history_p1['val_iou'][-1] > best_iou:
        best_iou = history_p1['val_iou'][-1]
        torch.save(model.state_dict(), CKPT)
        print(f'  → Saved (IoU: {best_iou:.4f})')

print(f'\nPhase 1 complete. Best IoU: {best_iou:.4f}')

---

## Phase 2 — Partial Backbone Unfreeze (20 epochs)

Last 2 transformer blocks + norm layer unfrozen. Differential LR: backbone=1e-5, neck+decoder=5e-5. Gradient clipping max_norm=1.0.

In [ ]:
# [9] Phase 2 setup — partial backbone unfreeze
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))

for p in model.parameters(): p.requires_grad = False
for p in model.backbone.blocks[-2:].parameters(): p.requires_grad = True
for p in model.backbone.norm.parameters(): p.requires_grad = True
for p in model.neck.parameters(): p.requires_grad = True
for p in model.decoder.parameters(): p.requires_grad = True

LR_BB_P2  = 1e-5
LR_DEC_P2 = 5e-5
NUM_EP_P2 = 20

bb_params  = [p for p in model.backbone.parameters() if p.requires_grad]
dec_params = [p for p in list(model.neck.parameters()) +
                         list(model.decoder.parameters()) if p.requires_grad]
optimizer_p2 = torch.optim.AdamW([
    {'params': bb_params,  'lr': LR_BB_P2},
    {'params': dec_params, 'lr': LR_DEC_P2},
], weight_decay=1e-4)
scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p2, T_max=NUM_EP_P2, eta_min=1e-6)
print(f'Phase 2: {NUM_EP_P2} epochs | LR backbone={LR_BB_P2} | LR neck+decoder={LR_DEC_P2}')

In [ ]:
# [10] Phase 2 training loop
history_p2 = {'train_loss': [], 'val_loss': [], 'val_iou': []}

for epoch in range(1, NUM_EP_P2 + 1):
    model.train()
    model.backbone.eval()
    model.backbone.blocks[-2].train()
    model.backbone.blocks[-1].train()
    model.backbone.norm.train()

    ep_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'P2 Epoch {epoch:02d}/{NUM_EP_P2} [train]', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer_p2.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_p2.step()
        ep_loss += loss.item()
    scheduler_p2.step()
    history_p2['train_loss'].append(ep_loss / len(train_loader))

    model.eval()
    v_loss, v_iou_list = 0.0, []
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'P2 Epoch {epoch:02d}/{NUM_EP_P2} [val]', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            v_loss += criterion(preds, masks).item()
            iou = fire_iou(preds, masks)
            if iou is not None: v_iou_list.append(iou)
    history_p2['val_loss'].append(v_loss / len(val_loader))
    history_p2['val_iou'].append(np.mean(v_iou_list) if v_iou_list else 0.0)

    print(f'Phase 2 Epoch {epoch:02d} | Train: {history_p2["train_loss"][-1]:.4f} | '
          f'Val: {history_p2["val_loss"][-1]:.4f} | IoU: {history_p2["val_iou"][-1]:.4f}')

    if history_p2['val_iou'][-1] > best_iou:
        best_iou = history_p2['val_iou'][-1]
        torch.save(model.state_dict(), CKPT)
        print(f'  → Saved (IoU: {best_iou:.4f})')

print(f'\nTraining complete. Best IoU: {best_iou:.4f}')

---

## Evaluation

Threshold optimization and final patch-by-patch metrics.

In [ ]:
# [11] Threshold optimization
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()

print('Collecting probabilities...')
all_probs, all_targets = [], []
with torch.no_grad():
    for ip, mp in tqdm(zip(val_imgs, val_masks), total=len(val_imgs)):
        raw  = np.load(ip,  mmap_mode='r').astype(np.float32)
        mask = np.load(mp,  mmap_mode='r')
        img  = (raw[BAND_IDX]/10000.0 - PRITHVI_MEAN[:,None,None]) / PRITHVI_STD[:,None,None]
        img  = img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        mc   = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy()
        prob = model(torch.from_numpy(img).unsqueeze(0).to(DEVICE)).sigmoid().squeeze().cpu().numpy()
        all_probs.append(prob.ravel())
        all_targets.append(mc.ravel())

all_probs   = np.concatenate(all_probs).astype(np.float32)
all_targets = np.concatenate(all_targets).astype(np.uint8)

thresholds = np.arange(0.30, 0.85, 0.025)
sweep = []
for t in thresholds:
    preds = (all_probs > t).astype(np.int32)
    tp = int(((preds==1)&(all_targets==1)).sum())
    fp = int(((preds==1)&(all_targets==0)).sum())
    fn = int(((preds==0)&(all_targets==1)).sum())
    iou  = tp/(tp+fp+fn+1e-6)
    prec = tp/(tp+fp+1e-6)
    rec  = tp/(tp+fn+1e-6)
    f1   = 2*prec*rec/(prec+rec+1e-6)
    sweep.append((t, iou, f1, prec, rec))

sweep = np.array(sweep)
bi = sweep[:,1].argmax()
THRESHOLD_OPT = float(sweep[bi, 0])
print(f'Optimal threshold: {THRESHOLD_OPT:.3f}  IoU={sweep[bi,1]:.4f}  F1={sweep[bi,2]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(sweep[:,0], sweep[:,1], label='IoU', color='steelblue', lw=2)
axes[0].plot(sweep[:,0], sweep[:,2], label='F1',  color='green', lw=2)
axes[0].plot(sweep[:,0], sweep[:,3], label='Precision', color='darkorange', lw=2, ls='--')
axes[0].plot(sweep[:,0], sweep[:,4], label='Recall',    color='crimson', lw=2, ls='--')
axes[0].axvline(THRESHOLD_OPT, color='green', ls='--', lw=1.5, label=f'Best (t={THRESHOLD_OPT:.3f})')
axes[0].set(xlabel='Threshold', ylabel='Score', title='Metrics vs Threshold')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].plot(sweep[:,4], sweep[:,3], color='steelblue', lw=2)
axes[1].scatter([sweep[bi,4]], [sweep[bi,3]], color='green', s=80, zorder=5,
                label=f't={THRESHOLD_OPT:.3f}  F1={sweep[bi,2]:.3f}')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.suptitle('Threshold Optimization — Prithvi-100M v1.5 | Corrientes val set', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS/'threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [12] Final evaluation + portfolio figure
print('Running patch-by-patch evaluation...')
all_samples = []
tp_all = fp_all = fn_all = 0
with torch.no_grad():
    for ip, mp in tqdm(zip(val_imgs, val_masks), total=len(val_imgs)):
        raw  = np.load(ip,  mmap_mode='r').astype(np.float32)
        mask = np.load(mp,  mmap_mode='r')
        img  = (raw[BAND_IDX]/10000.0 - PRITHVI_MEAN[:,None,None]) / PRITHVI_STD[:,None,None]
        img  = img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        mc   = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy()
        prob = model(torch.from_numpy(img).unsqueeze(0).to(DEVICE)).sigmoid().squeeze().cpu().numpy()
        pred = (prob > THRESHOLD_OPT).astype(np.uint8)
        tp = int(((pred==1)&(mc==1)).sum())
        fp = int(((pred==1)&(mc==0)).sum())
        fn = int(((pred==0)&(mc==1)).sum())
        iou = tp/(tp+fp+fn+1e-6)
        tp_all+=tp; fp_all+=fp; fn_all+=fn
        rgb = raw[[2,1,0], T:T+IMG_SIZE, T:T+IMG_SIZE]
        rgb = (rgb/(np.percentile(rgb,98)+1e-6)*255).clip(0,255).astype(np.uint8).transpose(1,2,0)
        all_samples.append((iou,rgb,mc,pred,tp,fp,fn,mc.sum()/(IMG_SIZE**2)))

g_iou  = tp_all/(tp_all+fp_all+fn_all+1e-6)
g_prec = tp_all/(tp_all+fp_all+1e-6)
g_rec  = tp_all/(tp_all+fn_all+1e-6)
g_f1   = 2*g_prec*g_rec/(g_prec+g_rec+1e-6)
print(f'IoU={g_iou:.4f}  F1={g_f1:.4f}  Prec={g_prec:.4f}  Rec={g_rec:.4f}')

vis = [s for s in all_samples if 0.08<=s[7]<=0.60]
vis.sort(key=lambda x: x[0])
n = len(vis)
picks = [('BEST',vis[int(n*.97)]),('MED #1',vis[int(n*.62)]),
         ('MED #2',vis[int(n*.38)]),('WORST',vis[int(n*.03)])]

def overlay(mask, pred):
    rgba = np.zeros((*mask.shape,4), dtype=np.float32)
    rgba[(pred==1)&(mask==1)] = [0.05,0.82,0.12,0.30]
    rgba[(pred==1)&(mask==0)] = [1.00,0.52,0.00,0.62]
    rgba[(pred==0)&(mask==1)] = [0.88,0.02,0.05,0.62]
    return rgba

fig = plt.figure(figsize=(15,7.5), facecolor='white')
fig.suptitle('Wildfire Burn Scar Detection from Sentinel-2 Satellite Imagery',
             fontsize=14, fontweight='bold', y=0.97)
fig.text(0.5,0.92,'PyTorch  |  Prithvi-100M (IBM/NASA)  |  6-channel Sentinel-2 L2A  |  dNBR labels  |  Corrientes, Argentina 2022',
         ha='center', fontsize=8.5, color='#444')
gs  = gridspec.GridSpec(1,2,figure=fig,left=0.02,right=0.97,top=0.89,bottom=0.09,wspace=0.05,width_ratios=[2.1,1])
gs_p = gridspec.GridSpecFromSubplotSpec(2,2,subplot_spec=gs[0],hspace=0.03,wspace=0.03)
lcol = {'BEST':'#097a27','MED #1':'#b86e00','MED #2':'#b86e00','WORST':'#a30000'}
for (label,s),(r,c) in zip(picks,[(0,0),(0,1),(1,0),(1,1)]):
    iou,rgb,mc,pred = s[0],s[1],s[2],s[3]
    ax = fig.add_subplot(gs_p[r,c])
    ax.imshow(rgb,interpolation='bilinear'); ax.imshow(overlay(mc,pred),interpolation='none')
    ax.text(0.012,0.985,f'{label}  |  Patch IoU = {iou:.3f}',transform=ax.transAxes,
            va='top',ha='left',fontsize=8.5,fontweight='bold',color='white',
            bbox=dict(facecolor=lcol[label],edgecolor='none',pad=2.5,alpha=0.88))
    ax.axis('off')
gs_r = gridspec.GridSpecFromSubplotSpec(2,1,subplot_spec=gs[1],hspace=0.44)
ax_b = fig.add_subplot(gs_r[0])
ep_all = list(range(1, len(history_p1['val_iou'])+len(history_p2['val_iou'])+1))
iou_all = history_p1['val_iou'] + history_p2['val_iou']
loss_all = history_p1['val_loss'] + history_p2['val_loss']
ax_b.plot(ep_all, loss_all, color='steelblue', lw=1.5, label='Val loss')
ax_b.plot(ep_all, iou_all,  color='#1aaa44',   lw=1.5, label='Val IoU')
ax_b.axvline(len(history_p1['val_iou']), color='#888', ls=':', lw=1.2, label='Phase 1→2')
ax_b.set_title('Training convergence (45 epochs, Colab A100)', fontsize=8.5, fontweight='bold')
ax_b.set_xlabel('Epoch', fontsize=7.5); ax_b.tick_params(labelsize=7)
ax_b.grid(alpha=0.2); ax_b.legend(fontsize=7, frameon=False); ax_b.set_ylim(bottom=0.2)
ax_m = fig.add_subplot(gs_r[1])
names=['Recall','Precision','F1','IoU']; values=[g_rec,g_prec,g_f1,g_iou]
bars = ax_m.barh(names, values, color='#4a7fbf', height=0.48)
for bar,val in zip(bars,values):
    ax_m.text(val+0.012, bar.get_y()+bar.get_height()/2, f'{val:.3f}',
              va='center', fontsize=9.5, fontweight='bold', color='#111')
ax_m.set_xlim(0,1.0)
ax_m.set_title(f'Global validation metrics  (n={len(val_imgs):,} patches)', fontsize=8.5, fontweight='bold')
ax_m.tick_params(labelsize=8.5); ax_m.grid(axis='x', alpha=0.2)
ax_m.spines[['top','right','left']].set_visible(False)
fig.legend(handles=[
    mpatches.Patch(color=(0.05,0.82,0.12), label='True Positive'),
    mpatches.Patch(color=(1.00,0.52,0.00), label='False Positive'),
    mpatches.Patch(color=(0.88,0.02,0.05), label='False Negative'),
], loc='lower left', ncol=3, fontsize=8, frameon=False, bbox_to_anchor=(0.02,0.005))
fig.text(0.5,0.012,
         'Prithvi-EO-1.0-100M (IBM/NASA)  |  Multi-scale neck (layers 2,5,8,11)  |  '
         'Labels: dNBR > 0.10  |  Corrientes, Argentina 2022  |  ~900,000 ha burned',
         ha='center', fontsize=7.5, color='#555')
plt.savefig(RESULTS/'validation_overview.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {RESULTS}/validation_overview.png')